# 📊 Data Collection and Preparation

## 1. Datasets Used

This project combines multiple datasets from Eurostat to construct a comprehensive panel dataset capturing housing market dynamics across EU countries. The datasets were selected to reflect both demand- and supply-side drivers of housing prices, as well as broader macroeconomic conditions.

### Core datasets (quarterly or high-frequency)

* **House Price Index (HPI)**

  * Source: Eurostat
  * Description: Quarterly residential property price index (base year 2015 = 100)
  * Role: Main target variable

* **Harmonised Index of Consumer Prices (HICP)**

  * Source: Eurostat
  * Description: Monthly inflation index (aggregated to quarterly)
  * Role: Captures inflation and cost dynamics

* **Gross Domestic Product (GDP)**

  * Source: Eurostat
  * Description: Real GDP index (chain-linked volumes)
  * Role: Proxy for economic activity and demand

* **Unemployment Rate**

  * Source: Eurostat
  * Description: Percentage of labour force unemployed (seasonally adjusted)
  * Role: Labour market conditions

* **Building Permits**

  * Source: Eurostat
  * Description: Number of residential dwelling permits (index, base 2021 = 100)
  * Role: Proxy for housing supply

---

### Optional datasets (annual, structural variables)

* **Old-age Dependency Ratio**

  * Measures the ratio of elderly population to working-age population

* **Population Density**

  * Measures population per square kilometre

These variables capture long-term demographic and structural differences across countries and are primarily used for clustering and descriptive analysis.

---

## 2. Key Data Processing Decisions

### 2.1 Country selection

The analysis focuses exclusively on **EU-27 countries**, ensuring:

* consistency across datasets
* comparability of economic structures
* sufficient data availability

A standardized mapping between country names and ISO country codes was implemented to ensure correct merging across datasets.

---

### 2.2 Frequency alignment (monthly and annual to quarterly)

The datasets were originally provided at different frequencies:

* Monthly (HICP)
* Quarterly (HPI, GDP, unemployment, permits)
* Annual (demographic variables)

To create a consistent panel:

* **Monthly → Quarterly**
  Monthly HICP values were aggregated by taking the average within each quarter.

* **Annual → Quarterly**
  Annual variables were expanded into quarterly frequency by assigning the same value to all four quarters of the corresponding year.

This approach assumes that structural variables evolve slowly and remain approximately constant within a year, which is appropriate for demographic indicators.

---

### 2.3 Variable selection

For each dataset, specific filters were applied to ensure consistency and interpretability:

* **HPI**: Nominal quarterly index (2015 = 100)
* **HICP**: Total inflation index (2015 = 100)
* **GDP**: Real GDP (chain-linked volumes, seasonally adjusted)
* **Unemployment**: Total population, 15–74 years, seasonally adjusted
* **Building permits**:

  * Number of dwellings
  * Residential buildings (excluding communal residences)
  * Index (2021 = 100)
    This specification was chosen based on data coverage and economic relevance

---

### 2.4 Handling missing data

Different strategies were used depending on the nature of the variable:

* **Core time-series variables (growth rates)**
  Missing values arise naturally from lag and target construction (first and last observations) and were removed.

* **Unemployment rate**
  Missing values were forward-filled within each country, reflecting the relatively slow-moving nature of labour market conditions.

* **Demographic variables (optional)**
  Forward-filled after expansion, but early missing values were retained.

---

### 2.5 Feature engineering

To capture dynamic relationships, several transformations were applied:

* **Quarter-on-quarter growth rates** were computed for:

  * House prices
  * Inflation
  * GDP
  * Building permits

* **Target variable**

  * Defined as next-quarter housing price growth

* **Lagged variables (t-1, t-2)**

  * Included for all key predictors
  * Capture temporal dependencies and market inertia

* **Eurozone indicators**

  * Current membership
  * Time-varying membership based on adoption year

---

## 3. Final Dataset

The final dataset is a **panel time series dataset** at the country–quarter level.

### Structure

* **Observations**: 2007 (full dataset), 1196 (model-ready dataset)
* **Countries**: 26 EU countries
* **Time span**: ~2005–2025 (varying slightly by country)

---

### Variable groups

#### Core variables (used for modeling)

* Housing:

  * `hpi_qoq_pct` (housing price growth)
* Macroeconomic:

  * `gdp_qoq_pct`
  * `hicp_qoq_pct`
  * `unemployment_rate`
* Supply:

  * `permits_qoq_pct`
* Institutional:

  * `eurozone_member_timevarying`

#### Lagged features

* All core variables include 1- and 2-quarter lags

#### Target

* `target_hpi_next_q`: next-quarter housing price growth

---

#### Optional variables

* `old_age_dependency_ratio`
* `population_density`

Used for:

* clustering analysis
* cross-country structural comparison

---

### Final modeling dataset

The final modeling dataset:

* contains no missing values
* includes lagged predictors
* preserves temporal ordering
* is suitable for both regression and machine learning models

---

## 4. Summary

The resulting dataset combines:

* macroeconomic indicators
* housing supply metrics
* demographic structure
* institutional factors

into a unified framework that allows for:

* forecasting housing price dynamics
* comparing country behavior
* identifying structural differences across European housing markets

This integrated approach provides a strong foundation for both predictive modeling and clustering analysis.


In [ ]:
import pandas as pd
import numpy as np
from io import StringIO
import requests


# ============================================================
# 1. LOAD DATA
# ============================================================
base_path = "./data"

hpi     = pd.read_csv(f"{base_path}/house_price_index.csv", low_memory=False)
hicp    = pd.read_csv(f"{base_path}/Harmonised index of consumer prices (HICP).csv", low_memory=False)
unemp   = pd.read_csv(f"{base_path}/unemployment.csv", low_memory=False)
gdp     = pd.read_csv(f"{base_path}/Gross domestic product (GDP) and main components (output, expenditure and income).csv", low_memory=False)
permits = pd.read_csv(f"{base_path}/building_permits.csv", low_memory=False)
old_age = pd.read_csv(f"{base_path}/Old-age-dependency ratio.csv", low_memory=False)
popdens = pd.read_csv(f"{base_path}/Population density.csv", low_memory=False)
lt_raw  = pd.read_csv(f"{base_path}/long_term_rates.csv", low_memory=False)


# mir data we have to get from api:
# Country codes available in MIR (eurozone members only - non-euro countries won't have this)
eurozone_mir = ["AT","BE","CY","DE","EE","ES","FI","FR","GR","HR","IE",
                "IT","LT","LU","LV","MT","NL","PT","SI","SK"]

dfs = []
for cc in eurozone_mir:
    url = (
        f"https://data-api.ecb.europa.eu/service/data/MIR/"
        f"M.{cc}.B.A2C.AM.R.A.2250.EUR.N"
        f"?format=csvdata&startPeriod=2005-01"
    )
    r = requests.get(url)
    if r.status_code == 200 and len(r.text) > 100:
        from io import StringIO
        df = pd.read_csv(StringIO(r.text))
        df["country_code"] = cc
        dfs.append(df)

mir = pd.concat(dfs, ignore_index=True)

# ============================================================
# 2. EU-27 COUNTRY MAP
# ============================================================
eu27 = {
    "Austria":"AT","Belgium":"BE","Bulgaria":"BG","Croatia":"HR","Cyprus":"CY",
    "Czechia":"CZ","Denmark":"DK","Estonia":"EE","Finland":"FI","France":"FR",
    "Germany":"DE","Greece":"EL","Hungary":"HU","Ireland":"IE","Italy":"IT",
    "Latvia":"LV","Lithuania":"LT","Luxembourg":"LU","Malta":"MT","Netherlands":"NL",
    "Poland":"PL","Portugal":"PT","Romania":"RO","Slovakia":"SK","Slovenia":"SI",
    "Spain":"ES","Sweden":"SE"
}
eu27_code_to_name = {v: k for k, v in eu27.items()}

def keep_eu27_by_name(df, country_col="geo"):
    df = df.copy()
    df = df[df[country_col].isin(eu27.keys())].copy()
    df["country_code"] = df[country_col].map(eu27)
    df["country"] = df[country_col]
    return df

# ============================================================
# 3. HOUSE PRICE INDEX
# ============================================================
hpi = hpi[hpi["unit"] == "I15_Q"].copy()
hpi = hpi[hpi["geo"].isin(eu27_code_to_name.keys())]
hpi = hpi.rename(columns={
    "geo": "country_code",
    "Geopolitical entity (reporting)": "country",
    "TIME_PERIOD": "quarter",
    "OBS_VALUE": "hpi"
})
hpi = hpi[["country_code","country","quarter","hpi"]].drop_duplicates()

# ============================================================
# 4. HICP → QUARTERLY
# ============================================================
hicp = hicp[(hicp["coicop18"]=="Total") & (hicp["unit"]=="Index, 2015=100")].copy()
hicp = keep_eu27_by_name(hicp)
hicp["date"] = pd.to_datetime(hicp["TIME_PERIOD"], format="%Y-%m")
hicp["quarter"] = hicp["date"].dt.to_period("Q").astype(str).str.replace("Q","-Q")
hicp_q = hicp.groupby(["country_code","country","quarter"], as_index=False)["OBS_VALUE"].mean()
hicp_q = hicp_q.rename(columns={"OBS_VALUE":"hicp_q"})

# ============================================================
# 5. UNEMPLOYMENT
# ============================================================
unemp = keep_eu27_by_name(unemp)
unemp = unemp[
    (unemp["sex"]=="Total") &
    (unemp["age"]=="From 15 to 74 years") &
    (unemp["unit"]=="Percentage of population in the labour force") &
    (unemp["s_adj"]=="Seasonally adjusted data, not calendar adjusted data")
]
unemp = unemp.rename(columns={"TIME_PERIOD":"quarter","OBS_VALUE":"unemployment_rate"})
unemp = unemp[["country_code","country","quarter","unemployment_rate"]]

# ============================================================
# 6. GDP
# ============================================================
gdp = keep_eu27_by_name(gdp)
gdp = gdp[
    (gdp["na_item"]=="Gross domestic product at market prices") &
    (gdp["s_adj"]=="Seasonally and calendar adjusted data") &
    (gdp["unit"]=="Chain linked volumes, index 2005=100")
]
gdp = gdp.rename(columns={"TIME_PERIOD":"quarter","OBS_VALUE":"gdp_index"})
gdp = gdp[["country_code","country","quarter","gdp_index"]]

# ============================================================
# 7. BUILDING PERMITS
# ============================================================
permits = keep_eu27_by_name(permits)
permits = permits[
    (permits["indic_bt"]=="Building permits - number of dwellings") &
    (permits["cpa2_1"]=="Residential buildings, except residences for communities") &
    (permits["unit"]=="Index, 2021=100")
]
permits = permits.rename(columns={"TIME_PERIOD":"quarter","OBS_VALUE":"building_permits_index"})
permits = permits[["country_code","country","quarter","building_permits_index"]]

# ============================================================
# 8. ANNUAL → QUARTERLY (demographic variables)
# ============================================================
def annual_to_quarter(df, value_col):
    df = keep_eu27_by_name(df)
    df = df.rename(columns={"TIME_PERIOD":"year","OBS_VALUE":value_col})
    df["year"] = df["year"].astype(int)
    out = []
    for q in ["Q1","Q2","Q3","Q4"]:
        tmp = df.copy()
        tmp["quarter"] = tmp["year"].astype(str)+"-"+q
        out.append(tmp[["country_code","country","quarter",value_col]])
    return pd.concat(out)

old_age_q = annual_to_quarter(old_age, "old_age_dependency_ratio")
popdens_q = annual_to_quarter(popdens, "population_density")

# ============================================================
# 7b. MIR — ECB Mortgage Interest Rates (eurozone only)
# ============================================================
mir["OBS_VALUE"] = (
    mir["OBS_VALUE"].astype(str)
    .str.replace(",", ".", regex=False).str.strip()
)
mir["OBS_VALUE"] = pd.to_numeric(mir["OBS_VALUE"], errors="coerce")
mir["date"] = pd.to_datetime(mir["TIME_PERIOD"], format="%Y-%m")
mir["quarter"] = mir["date"].dt.to_period("Q").astype(str).str.replace("Q","-Q")
mir_q = (
    mir.groupby(["country_code","quarter"], as_index=False)["OBS_VALUE"]
    .mean()
    .rename(columns={"OBS_VALUE":"mortgage_rate_pct"})
)
mir_q = mir_q[mir_q["country_code"].isin(eu27.values())].copy()
mir_q = mir_q.sort_values(["country_code","quarter"])
mir_q["mortgage_rate_pct"] = mir_q.groupby("country_code")["mortgage_rate_pct"].transform(lambda x: x.ffill())

# ============================================================
# 7c. LONG-TERM INTEREST RATES — Eurostat IRT_LT_MCBY_Q
# ============================================================
lt_raw = lt_raw.rename(columns={"geo":"country","TIME_PERIOD":"quarter","OBS_VALUE":"lt_interest_rate"})
lt_raw = lt_raw[lt_raw["country"].isin(eu27.keys())].copy()
lt_raw["country_code"] = lt_raw["country"].map(eu27)
lt_rates_q = lt_raw[["country_code","quarter","lt_interest_rate"]].drop_duplicates(subset=["country_code","quarter"])
lt_rates_q = lt_rates_q.sort_values(["country_code","quarter"])
lt_rates_q["lt_interest_rate"] = lt_rates_q.groupby("country_code")["lt_interest_rate"].transform(lambda x: x.ffill())

# ============================================================
# 9. MERGE
# ============================================================
master = hpi.copy()

merge_dfs = [hicp_q, unemp, gdp, permits, old_age_q, popdens_q, mir_q, lt_rates_q]
for df in merge_dfs:
    df_clean = df.drop(columns=["country"], errors="ignore")
    master = master.merge(df_clean, on=["country_code","quarter"], how="left")

# ============================================================
# 10. TIME FEATURES
# ============================================================
master["year"] = master["quarter"].str[:4].astype(int)
master["quarter_num"] = master["quarter"].str[-1].astype(int)

# ============================================================
# 11. EUROZONE FLAGS
# ============================================================
current_eurozone = {"AT","BE","BG","HR","CY","EE","FI","FR","DE","EL","IE","IT",
                    "LV","LT","LU","MT","NL","PT","SK","SI","ES"}
master["eurozone_member_current"] = master["country_code"].isin(current_eurozone).astype(int)

adoption_year = {
    "AT":1999,"BE":1999,"BG":2026,"HR":2023,"CY":2008,"EE":2011,"FI":1999,
    "FR":1999,"DE":1999,"EL":2001,"IE":1999,"IT":1999,"LV":2014,"LT":2015,
    "LU":1999,"MT":2008,"NL":1999,"PT":1999,"SK":2009,"SI":2007,"ES":1999
}
master["eurozone_member_timevarying"] = master.apply(
    lambda r: int(r["country_code"] in adoption_year and r["year"] >= adoption_year[r["country_code"]]),
    axis=1
)

# ============================================================
# 12. SORT + GROWTH RATES + TARGET
# ============================================================
master = master.sort_values(["country_code","quarter"])

master["hpi_qoq_pct"]     = master.groupby("country_code")["hpi"].pct_change()*100
master["hicp_qoq_pct"]    = master.groupby("country_code")["hicp_q"].pct_change()*100
master["gdp_qoq_pct"]     = master.groupby("country_code")["gdp_index"].pct_change()*100
master["permits_qoq_pct"] = master.groupby("country_code")["building_permits_index"].pct_change()*100
master["target_hpi_next_q"] = master.groupby("country_code")["hpi_qoq_pct"].shift(-1)

# ============================================================
# 13. FORWARD FILL STRUCTURAL + UNEMPLOYMENT
# ============================================================
for col in ["old_age_dependency_ratio","population_density"]:
    master[col] = master.groupby("country_code")[col].transform(lambda x: x.ffill())
master["unemployment_rate"] = master.groupby("country_code")["unemployment_rate"].transform(lambda x: x.ffill())
master["mortgage_rate_pct"] = master.groupby("country_code")["mortgage_rate_pct"].transform(lambda x: x.ffill())
master["lt_interest_rate"]  = master.groupby("country_code")["lt_interest_rate"].transform(lambda x: x.ffill())

# ============================================================
# 14. BUILD MODEL DATASET
# ============================================================
required_cols = [
    "hpi_qoq_pct","hicp_qoq_pct","gdp_qoq_pct",
    "permits_qoq_pct","target_hpi_next_q"
]
model_data = master.dropna(subset=required_cols).copy()

# ============================================================
# 15. ADD LAGS
# ============================================================
lag_cols = [
    "hpi_qoq_pct","hicp_qoq_pct","gdp_qoq_pct","permits_qoq_pct",
    "unemployment_rate","mortgage_rate_pct","lt_interest_rate"
]
for col in lag_cols:
    model_data[f"{col}_lag1"] = model_data.groupby("country_code")[col].shift(1)
    model_data[f"{col}_lag2"] = model_data.groupby("country_code")[col].shift(2)

# Drop rows where core lag columns are missing (first 2 obs per country)
core_lag_required = [f"{c}_lag1" for c in ["hpi_qoq_pct","hicp_qoq_pct","gdp_qoq_pct","permits_qoq_pct","unemployment_rate"]]
model_data = model_data.dropna(subset=core_lag_required)

# ============================================================
# 16. UNEMPLOYMENT QOQ CHANGE
# ============================================================
model_data["unemployment_qoq_change"] = model_data.groupby("country_code")["unemployment_rate"].diff()

# ============================================================
# 17. FINAL CHECKS
# ============================================================
print("MASTER:", master.shape)
print("MODEL:", model_data.shape)
print("\nColumn list:")
print(model_data.columns.tolist())
print("\nMissing values:")
print(model_data.isna().sum()[model_data.isna().sum() > 0])
print("\nCoverage check (non-null counts):")
print(model_data.groupby("country_code")[["mortgage_rate_pct","lt_interest_rate"]].count())

MASTER: (2007, 21)
MODEL: (1704, 36)

Column list:
['country_code', 'country', 'quarter', 'hpi', 'hicp_q', 'unemployment_rate', 'gdp_index', 'building_permits_index', 'old_age_dependency_ratio', 'population_density', 'mortgage_rate_pct', 'lt_interest_rate', 'year', 'quarter_num', 'eurozone_member_current', 'eurozone_member_timevarying', 'hpi_qoq_pct', 'hicp_qoq_pct', 'gdp_qoq_pct', 'permits_qoq_pct', 'target_hpi_next_q', 'hpi_qoq_pct_lag1', 'hpi_qoq_pct_lag2', 'hicp_qoq_pct_lag1', 'hicp_qoq_pct_lag2', 'gdp_qoq_pct_lag1', 'gdp_qoq_pct_lag2', 'permits_qoq_pct_lag1', 'permits_qoq_pct_lag2', 'unemployment_rate_lag1', 'unemployment_rate_lag2', 'mortgage_rate_pct_lag1', 'mortgage_rate_pct_lag2', 'lt_interest_rate_lag1', 'lt_interest_rate_lag2', 'unemployment_qoq_change']

Missing values:
old_age_dependency_ratio    508
population_density          404
mortgage_rate_pct           561
lt_interest_rate             44
hpi_qoq_pct_lag2              6
hicp_qoq_pct_lag2             6
gdp_qoq_pct_lag